# 05 LLM Strategy Generation

This notebook demonstrates Deepseek-based strategy generation for credit lifecycle scenarios:

- Configure `.env` for Deepseek API access
- Validate the `DeepseekClient` call flow
- Generate strategies for 10 test users
- Run compliance re-checking
- Save the test results to `docs/`


In [ ]:
from pathlib import Path
import json
import sys
from datetime import datetime

import pandas as pd

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
SRC_DIR = PROJECT_ROOT / 'src'
DOCS_DIR = PROJECT_ROOT / 'docs'
ENV_PATH = PROJECT_ROOT / '.env'
ENV_EXAMPLE_PATH = PROJECT_ROOT / '.env.example'
DOCS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from llm import CreditStrategyGenerator, DeepseekClient


## Step 1: Configure `.env`

Before running the API test, make sure the project root contains a valid `.env` file.

Example:

```bash
cp .env.example .env
# Then replace DEEPSEEK_API_KEY with your real key
```


In [ ]:
# Optional helper: create `.env` from `.env.example` if it does not exist.
if not ENV_PATH.exists() and ENV_EXAMPLE_PATH.exists():
    ENV_PATH.write_text(ENV_EXAMPLE_PATH.read_text(encoding='utf-8'), encoding='utf-8')
    print('Created .env from .env.example. Please edit it with a real API key before calling the API.')
else:
    print('.env already exists or .env.example is missing.')


In [ ]:
# Step 2: Test DeepseekClient connectivity
client = None
client_test_result = {}

try:
    client = DeepseekClient()
    ping_reply = client.generate('请用一句中文确认你已成功连接，并返回“连接成功”。', temperature=0.1, max_tokens=60)
    client_test_result = {'status': 'success', 'reply': ping_reply}
except Exception as exc:
    client_test_result = {'status': 'failed', 'error': str(exc)}

print(json.dumps(client_test_result, ensure_ascii=False, indent=2))


In [ ]:
# Step 3: Prepare 10 test users across multiple scenarios
test_users = [
    {'user_id': 'U001', 'name': '张三', 'preloan_risk_label': 0, 'loan_balance': 18000, 'annual_inc': 180000, 'dti': 18.5, 'payment_history': '历史还款稳定，无逾期'},
    {'user_id': 'U002', 'name': '李四', 'preloan_risk_label': 0, 'loan_balance': 26000, 'annual_inc': 220000, 'dti': 22.1, 'payment_history': '近12个月正常还款'},
    {'user_id': 'U003', 'name': '王五', 'preloan_risk_label': 1, 'loan_balance': 15000, 'annual_inc': 120000, 'dti': 39.8, 'payment_history': '近期存在轻微还款波动'},
    {'user_id': 'U004', 'name': '赵六', 'preloan_risk_label': 1, 'loan_balance': 21000, 'annual_inc': 140000, 'dti': 44.2, 'payment_history': '有一次延后还款记录'},
    {'user_id': 'U005', 'name': '孙七', 'preloan_risk_label': 2, 'loan_balance': 32000, 'annual_inc': 100000, 'dti': 58.3, 'payment_history': '近期出现逾期苗头'},
    {'user_id': 'U006', 'name': '周八', 'preloan_risk_label': 2, 'loan_balance': 28000, 'annual_inc': 110000, 'dti': 52.0, 'payment_history': '连续两期还款压力偏大'},
    {'user_id': 'U007', 'name': '吴九', 'preloan_risk_label': 3, 'loan_balance': 40000, 'annual_inc': 90000, 'dti': 66.7, 'payment_history': '已发生明显逾期'},
    {'user_id': 'U008', 'name': '郑十', 'preloan_risk_label': 3, 'loan_balance': 46000, 'annual_inc': 85000, 'dti': 71.3, 'payment_history': '存在连续逾期记录'},
    {'user_id': 'U009', 'name': '钱十一', 'preloan_risk_label': 4, 'loan_balance': 52000, 'annual_inc': 78000, 'dti': 79.1, 'payment_history': '高风险客户，需协商处理'},
    {'user_id': 'U010', 'name': '孙十二', 'preloan_risk_label': 4, 'loan_balance': 60000, 'annual_inc': 72000, 'dti': 82.5, 'payment_history': '疑似坏账，需协商分期'},
]

pd.DataFrame(test_users)


In [ ]:
# Step 4: Generate strategies and run compliance checks
generator = None
strategy_results = []

try:
    generator = CreditStrategyGenerator()
    for user in test_users:
        result = generator.generate_strategy(user)
        strategy_results.append({
            'user_id': user['user_id'],
            'name': user['name'],
            'risk_label': user['preloan_risk_label'],
            'scenario': result['scenario'],
            'generated_content': result['generated_content'],
            'final_content': result['final_content'],
        })
except Exception as exc:
    strategy_results.append({
        'user_id': 'SYSTEM',
        'name': 'SYSTEM',
        'risk_label': -1,
        'scenario': 'ERROR',
        'generated_content': '',
        'final_content': f'LLM workflow failed: {exc}',
    })

result_df = pd.DataFrame(strategy_results)
display(result_df)


In [ ]:
# Step 5: Save test results to docs/
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
json_output_path = DOCS_DIR / f'llm_strategy_test_results_{timestamp}.json'
markdown_output_path = DOCS_DIR / f'llm_strategy_test_results_{timestamp}.md'

json_output_path.write_text(
    json.dumps(strategy_results, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

markdown_lines = [
    '# LLM Strategy Generation Test Results',
    '',
    f'- Generated at: {timestamp}',
    f"- Client test status: {client_test_result.get('status', 'unknown')}",
    '',
]

for item in strategy_results:
    markdown_lines.append(f"## {item['user_id']} - {item['name']}")
    markdown_lines.append(f"- Scenario: {item['scenario']}")
    markdown_lines.append(f"- Risk Label: {item['risk_label']}")
    markdown_lines.append('')
    markdown_lines.append('### Final Content')
    markdown_lines.append(item['final_content'])
    markdown_lines.append('')

markdown_output_path.write_text('\n'.join(markdown_lines), encoding='utf-8')

print('Saved JSON results to:', json_output_path)
print('Saved Markdown results to:', markdown_output_path)
